## Change the event time

In [27]:
import pandas as pd
import numpy as np

def transform_event_time(df,
                         cohort_col="cohort",
                         event_col="event_time",
                         target_months=[1, 2, 3, 11, 12],
                         pre_start=-2):   # <- 可選 -1 或 -2

    df = df.copy()

    # 嘗試多種 cohort format
    formats = ["%b-%y", "%Y-%m", "%Y/%m", "%Y%m"]

    cohort_date = None

    for fmt in formats:
        try:
            cohort_date = pd.to_datetime(df[cohort_col], format=fmt)
            print(f"Use format: {fmt}")
            break
        except:
            pass

    # 全部失敗時自動推斷
    if cohort_date is None:
        cohort_date = pd.to_datetime(
            df[cohort_col],
            errors="coerce"
        )

    # cohort 月份
    df["cohort_month"] = cohort_date.dt.month

    # cohort + event_time 對應月份
    df["cohort_event_month"] = (
        (df["cohort_month"] + df[event_col] - 1) % 12
    ) + 1

    # 是否 target month
    df["target_month_flag"] = (
        df["cohort_event_month"]
        .isin(target_months)
        .astype(int)
    )

    # 重編 event time
    def reindex(g):

        g = g.copy()

        g["new_event_time"] = pd.NA

        # cohort = 0
        g.loc[g[event_col] == 0, "new_event_time"] = 0

        # cohort 前
        pre_idx = (
            g[
                (g[event_col] < 0) &
                (g["target_month_flag"] == 1)
            ]
            .sort_values(event_col, ascending=False)
            .index
        )

        g.loc[pre_idx, "new_event_time"] = list(
            range(
                pre_start,
                pre_start - len(pre_idx),
                -1
            )
        )

        # cohort 後
        post_idx = (
            g[
                (g[event_col] > 0) &
                (g["target_month_flag"] == 1)
            ]
            .sort_values(event_col)
            .index
        )

        g.loc[post_idx, "new_event_time"] = list(
            range(1, len(post_idx) + 1)
        )

        return g

    df = (
        df.groupby(cohort_col, group_keys=False)
        .apply(reindex)
    )

    # 轉整數
    df["new_event_time"] = df["new_event_time"].astype("Int64")

    return df

In [28]:
df = pd.read_csv("combine/low_dynamic_cohort_effect.csv")

In [29]:
df2 = transform_event_time(df, pre_start=-2)
df2
df2.to_csv("combine/low_dynamic_cohort_effect_reindexed.csv", index=False)

Use format: %Y-%m


In [30]:
import pandas as pd
import numpy as np

def collapse_cohort_month(
    df,
    cohort_range=range(4, 11),
    weight_col="n_treated"
):

    # 只取 cohort_month 4~10
    d = df[df["cohort_month"].isin(cohort_range)].copy()

    weighted_cols = [
        "effect",
        "se",
        "t_stat",
        "p_value",
        "ci_low",
        "ci_high",
        "n_control"
    ]

    # weighted average function
    def wavg(x, val_col):
        return np.average(
            x[val_col],
            weights=x[weight_col]
        )

    result = (
        d.groupby("new_event_time")
        .apply(
            lambda x: pd.Series({

                # weighted avg
                "effect": wavg(x, "coef"),
                "se": wavg(x, "std_error"),
                # "t_stat": wavg(x, "t_stat"),
                "p_value": wavg(x, "p_value"),
                "ci_low": wavg(x, "ci_lower"),
                "ci_high": wavg(x, "ci_upper"),
                "n_control": wavg(x, "n_treated"),

                # treated 直接加總
                "n_treated": x["n_treated"].sum(),
 
                # cohort label
                "cohort_group": "2025-04 to 2025-10"
            })
        )
        .reset_index()
    )

    return result

In [31]:
df_new = collapse_cohort_month(df2)
df_new

,new_event_time,effect,se,p_value,ci_low,ci_high,n_control,n_treated,cohort_group
0,-6,-0.290591,0.317442,0.584994,-0.912777,0.331595,97.910256,468,2025-04 to 2025-10
1,-5,-0.168478,0.222632,0.495599,-0.604836,0.267880,196.231920,802,2025-04 to 2025-10
2,-4,-0.250402,0.233287,0.270388,-0.707643,0.206840,196.231920,802,2025-04 to 2025-10
3,-3,-0.239326,0.231300,0.522805,-0.692673,0.214022,196.231920,802,2025-04 to 2025-10
4,-2,-0.191552,0.207309,0.270953,-0.597878,0.214774,196.231920,802,2025-04 to 2025-10
5,0,0.329092,0.187195,0.129017,-0.037811,0.695995,196.231920,802,2025-04 to 2025-10
6,1,0.358361,0.283798,0.294206,-0.197884,0.914606,97.910256,468,2025-04 to 2025-10
7,2,0.144989,0.329708,0.540475,-0.501239,0.791217,79.945122,328,2025-04 to 2025-10
8,3,-0.047006,0.433091,0.652992,-0.895865,0.801852,52.201970,203,2025-04 to 2025-10
9,4,0.135723,0.475671,0.774837,-0.796592,1.068039,46.769784,139,2025-04 to 2025-10


In [32]:
df_new.to_csv("combine/low_dynamic_cohort_effect_reindexed_4-10.csv", index=False)